# L200 · Deploy the agent as a Databricks App

The recommended path is the CLI with Databricks Asset Bundles (see the **deploy** skill):

```bash
uv run preflight                 # start locally, send a test request
databricks bundle validate
databricks bundle deploy
databricks bundle run akzo_capabilities_agent
```

This notebook is the SDK based alternative for environments where you deploy from inside the workspace. Everything is parametrized with widgets, so nothing is tied to one workspace, user, or app name.

In [ ]:
dbutils.widgets.text("app_name", "agent-akzo-capabilities", "Databricks App name")
dbutils.widgets.text("source_code_path", "", "Workspace source path (the L200-capabilities folder)")

app_name = dbutils.widgets.get("app_name").strip()
source_code_path = dbutils.widgets.get("source_code_path").strip()

if not source_code_path:
    # Default to this notebook's workspace folder's parent (the tier root).
    import os
    here = os.path.dirname(os.path.dirname(os.getcwd()))
    source_code_path = here
    print(f"source_code_path not set; defaulting to {source_code_path}")

print(f"App name:    {app_name}")
print(f"Source path: {source_code_path}")

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import AppDeployment

w = WorkspaceClient()

# Deploy the app (creates a new deployment from workspace source).
deployment = w.apps.deploy_and_wait(
    app_name=app_name,
    app_deployment=AppDeployment(
        source_code_path=source_code_path,
        mode="SNAPSHOT",
    ),
)
print("✓ Deployment complete")
print(f"  Status:        {deployment.status.state}")
print(f"  Deployment ID: {deployment.deployment_id}")

## After deploy

Grant the app's service principal access to the Lakebase memory + Action Plane schemas:

```bash
SP=$(databricks apps get <app-name> --output json | jq -r '.service_principal_client_id')
uv run python scripts/grant_lakebase_permissions.py $SP --memory-type openai --instance-name <your-instance>
```

See the **deploy** and **lakebase-setup** skills for the full flow.